# 世界通史时空图谱 · 国别演变 & 专题导览语音扩展工坊（Colab）

为 30 个精编国家的历代阶段（85 段）与 10 大专题导览全站点（约 72 段）一次性并发合成中文纪录片语音（`edge-tts`，标准 CPU 运行时仅需约 **1 分钟**，零消耗 GPU 计算点）。

```bash
npm run media:import -- ~/Downloads/atlas-extras-audio-pack.zip
```

In [ ]:
#@title 1. 设置工作目录
import os

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_SUBDIR = "world-history-atlas/extras-audio"  #@param {type:"string"}

WORK_DIR = "/content/atlas-extras-audio"
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK_DIR = os.path.join("/content/drive/MyDrive", DRIVE_SUBDIR)
    except Exception as exc:
        print(f"Drive 挂载跳过: {exc}")

os.makedirs(WORK_DIR, exist_ok=True)
print("📁 工作目录:", WORK_DIR)

In [ ]:
#@title 2. 安装依赖与参数配置 (edge-tts + ffmpeg)
VOICE_PRESET = "zh-CN-YunjianNeural"  #@param ["zh-CN-YunjianNeural", "zh-CN-YunxiNeural", "zh-CN-XiaoxiaoNeural"]
SPEECH_RATE = "+6%"  #@param {type:"string"}
MP3_BITRATE = "32k"  #@param ["24k", "32k", "48k"]
CONCURRENCY = 6  #@param {type:"integer"}

!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install --upgrade edge-tts mutagen nest_asyncio
print(f"✅ 依赖就绪 | 音色: {VOICE_PRESET} | 并发: {CONCURRENCY}")

In [ ]:
#@title 3. 拉取国别与专题语音脚本清单 (extras-audio-scripts.json)
import json, urllib.request

RAW_SCRIPTS_URL = "https://raw.githubusercontent.com/quanliulq-pixel/world-history-atlas/main/colab/extras-audio-scripts.json"  #@param {type:"string"}
LOCAL_SCRIPTS_PATH = os.path.join(WORK_DIR, "extras-audio-scripts.json")

urllib.request.urlretrieve(RAW_SCRIPTS_URL, LOCAL_SCRIPTS_PATH)
with open(LOCAL_SCRIPTS_PATH, "r", encoding="utf-8") as f:
    extras_catalog = json.load(f)

print("📊 清单统计:", extras_catalog.get("counts", {}))

In [ ]:
#@title 4. 并发批量合成并写入 manifest.json
import asyncio, datetime, subprocess, time
import nest_asyncio
import edge_tts
from mutagen.mp3 import MP3

nest_asyncio.apply()

MANIFEST_PATH = os.path.join(WORK_DIR, "manifest.json")
manifest = {"version": 1, "engine": f"edge-tts:{VOICE_PRESET}", "items": {}}
items = extras_catalog["items"]

def get_duration(mp3_path: str) -> float:
    try:
        return round(float(MP3(mp3_path).info.length), 2)
    except Exception:
        return 0.0

async def synth_one(sem: asyncio.Semaphore, item: dict, stats: dict):
    rel_file = item["file"]
    abs_file = os.path.join(WORK_DIR, rel_file)
    os.makedirs(os.path.dirname(abs_file), exist_ok=True)

    async with sem:
        if not os.path.exists(abs_file):
            tmp_mp3 = abs_file + ".raw.mp3"
            for attempt in range(3):
                try:
                    comm = edge_tts.Communicate(item["script"], VOICE_PRESET, rate=SPEECH_RATE)
                    await comm.save(tmp_mp3)
                    subprocess.run(
                        ["ffmpeg", "-y", "-loglevel", "error", "-i", tmp_mp3, "-ac", "1", "-ar", "24000", "-b:a", MP3_BITRATE, abs_file],
                        check=True,
                    )
                    if os.path.exists(tmp_mp3):
                        os.remove(tmp_mp3)
                    break
                except Exception as exc:
                    if attempt == 2:
                        raise exc
                    await asyncio.sleep(1.2 * (attempt + 1))

    manifest["items"][item["key"]] = {
        "kind": item["kind"],
        "id": item["id"],
        "script": item["script"],
        "voice": VOICE_PRESET,
        "file": rel_file,
        "durationSec": get_duration(abs_file),
    }
    stats["done"] += 1
    if stats["done"] % 25 == 0 or stats["done"] == len(items):
        elapsed = time.time() - stats["start"]
        print(f"[{stats['done']}/{len(items)}] ✅ {item['title']} | 已耗时 {elapsed:.1f}s")

async def run_all():
    sem = asyncio.Semaphore(max(1, CONCURRENCY))
    stats = {"done": 0, "start": time.time()}
    await asyncio.gather(*(synth_one(sem, item, stats) for item in items))
    manifest["generatedAt"] = datetime.datetime.now(datetime.timezone.utc).isoformat().replace("+00:00", "Z")
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print(f"\n🎉 全部 {len(manifest['items'])} 段扩展语音合成完成！")

asyncio.get_event_loop().run_until_complete(run_all())

In [ ]:
#@title 5. 打包导出 atlas-extras-audio-pack.zip 并自动下载
import zipfile

ZIP_PATH = "/content/atlas-extras-audio-pack.zip"
file_count = 0

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(MANIFEST_PATH, "manifest.json")
    for entry in manifest["items"].values():
        rel = entry["file"]
        abs_path = os.path.join(WORK_DIR, rel)
        if os.path.exists(abs_path):
            zf.write(abs_path, rel)
            file_count += 1

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"📦 已打包 {file_count} 段扩展音频 -> {ZIP_PATH} ({size_mb:.2f} MB)")

try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception as exc:
    print(f"请手动下载 {ZIP_PATH}: {exc}")